In [ ]:
#  Load processed/preprocessed data
import json
import numpy as np
import pandas as pd
import os

df = pd.read_csv("../transaction_data.csv")
print(df.shape)
df.head()

In [ ]:
#  Compute config stats (thresholds + step counts map)
config = {}

# 95th percentile threshold for "large transaction"
config["large_threshold_p95"] = float(df["amount"].quantile(0.95))

# median for destination high balance flag
config["median_oldbalanceDest"] = float(df["oldbalanceDest"].median())

# step bin edges (same logic as your feature engineering)
max_step = int(df["step"].max())
bins = np.linspace(0, max_step, 7).tolist()
config["step_bins"] = bins

# step -> transaction count (velocity proxy)
step_counts = df.groupby("step").size().to_dict()
# keys must be str for json
config["step_counts"] = {str(int(k)): int(v) for k, v in step_counts.items()}

# median transactions in a step (for high velocity flag)
config["median_transactions_in_step"] = float(np.median(list(step_counts.values())))

config

In [ ]:
os.makedirs("../models", exist_ok=True)

out_path = "../models/feature_config.json"
with open(out_path, "w") as f:
    json.dump(config, f, indent=2)

print("Saved:", out_path)

In [ ]:
import json, os
import numpy as np
import pandas as pd

pre = pd.read_csv("../data/processed/preprocessed_data.csv")  # has type column ideally
eng = pd.read_csv("../data/processed/feature_engineered_data.csv")  # final training style

print("pre:", pre.shape)
print("eng:", eng.shape)

In [ ]:
config = {}

config["large_threshold_p95"] = float(pre["amount"].quantile(0.95))
config["median_oldbalanceDest"] = float(pre["oldbalanceDest"].median())

max_step = int(pre["step"].max())
config["step_bins"] = np.linspace(0, max_step, 7).tolist()

step_counts = pre.groupby("step").size().to_dict()
config["step_counts"] = {str(int(k)): int(v) for k, v in step_counts.items()}
config["median_transactions_in_step"] = float(np.median(list(step_counts.values())))

config


os.makedirs("../models", exist_ok=True)

with open("../models/feature_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved: ../models/feature_config.json")

In [ ]:
drop_cols = ["isFraud", "isFlaggedFraud"]  # same as training
X_cols = [c for c in eng.columns if c not in drop_cols]

with open("../models/feature_columns.json", "w") as f:
    json.dump(X_cols, f, indent=2)

print("Saved: ../models/feature_columns.json")
print("Total model features:", len(X_cols))